# OpenEnv SME Negotiator — GRPO Training on Colab
**OpenEnv Hackathon Submission** | [HF Space](https://huggingface.co/openenv) | [GitHub](https://github.com/SkandaGanesha1/OpenEnv_SME_Negotiator-2.o)

Train an SME treasury negotiation agent using GRPO with HF TRL. The agent learns to negotiate favorable payment terms with large buyers — importing training logic from the `sme_negotiator_env` package.

**Theme #1 — Multi-Agent Interactions** (Razorpay Fix-My-Itch #82.8: SME vs large-buyer payment-term negotiation).

| Component | Detail |
|-----------|--------|
| Environment | In-process `SMELiquidityEnvironment` (multi-deal, multi-period) |
| Training | This Colab notebook (T4 / A100 GPU) |
| Model | `Qwen/Qwen2.5-1.5B-Instruct` + LoRA |
| Framework | HF TRL v0.29+ GRPO with `rollout_func` + `generate_rollout_completions` |

This notebook follows the **canonical OpenEnv + TRL pattern** (same as [kube-sre-gym](https://github.com/sid-rp/kube-sre-gym) and [wordle.py](https://github.com/meta-pytorch/OpenEnv/blob/main/tutorial/examples/wordle.py)):
1. Install OpenEnv + TRL deps.
2. Smoke-test the in-process `SMELiquidityEnvironment`.
3. Score a deterministic heuristic baseline.
4. GRPO fine-tune with **five verifiable reward functions** (total · solvency · NPV · compliance · format).
5. Plot reward curves and compare baseline vs trained.
6. Optional: push to HF Hub.

## 1. Install dependencies

We pin TRL 0.29 because that is the latest release that exposes both `rollout_func=` and `trl.experimental.openenv.generate_rollout_completions`.

In [ ]:
%pip install -q -U pip
%pip install -q \
    "trl[vllm]>=0.29.0" \
    "vllm>=0.11.0" \
    "peft>=0.19.1" \
    "transformers>=4.45,<4.60" \
    "datasets" \
    "accelerate" \
    "matplotlib" \
    "numpy" \
    "huggingface_hub>=0.20" \
    "openenv-core"

In [ ]:
import os
from pathlib import Path

REPO_DIR = Path("OpenEnv_SME_Negotiator-2.o")
if not REPO_DIR.exists():
    !git clone -q https://github.com/SkandaGanesha1/OpenEnv_SME_Negotiator-2.o.git
%cd $REPO_DIR
%pip install -q -e .[rl]

## 2. Configuration

All LLM inference is routed through the **Hugging Face Router** (`https://router.huggingface.co/v1`). Add `HF_TOKEN` in Colab Secrets (the key icon in the left sidebar) so the trained policy can call the router during evaluation.

In [ ]:
import os
from pathlib import Path

# Silence experimental-API warnings (same as kube-sre-gym).
os.environ.setdefault("TRL_EXPERIMENTAL_SILENCE", "1")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

# Hugging Face Router (canonical for this hackathon).
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["OPENAI_API_KEY"] = hf_token
        os.environ["OPENAI_BASE_URL"] = "https://router.huggingface.co/v1"
        os.environ["API_BASE_URL"]   = "https://router.huggingface.co/v1"
except Exception:
    pass

MODEL_NAME    = "Qwen/Qwen2.5-1.5B-Instruct"   # fits a free-tier T4
TASK_NAME     = "liquidity-stress-medium"
DIFFICULTY    = "medium"
TOTAL_PERIODS = 2
MAX_TURNS     = 15          # max env steps per rollout episode
MAX_NEW_TOKENS = 256        # per LLM turn; action JSON is short but needs room for reasoning
DATASET_SIZE  = 32          # small for a Colab demo; raise to 128+ for real training
NUM_GENERATIONS = 4         # GRPO group size (>= 2 required)
TRAIN_STEPS   = 25
MAX_TOTAL_TOKENS = 4096     # cap to prevent OOM during backward pass
OUTPUT_DIR    = Path("outputs/grpo_sme_liquidity_colab")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"MODEL_NAME       = {MODEL_NAME}")
print(f"TASK_NAME        = {TASK_NAME} (difficulty={DIFFICULTY}, total_periods={TOTAL_PERIODS})")
print(f"DATASET_SIZE     = {DATASET_SIZE}, NUM_GENERATIONS = {NUM_GENERATIONS}, TRAIN_STEPS = {TRAIN_STEPS}")
print(f"MAX_TURNS        = {MAX_TURNS}, MAX_NEW_TOKENS = {MAX_NEW_TOKENS}")
print(f"OUTPUT_DIR       = {OUTPUT_DIR}")

## 3. Smoke-test the environment (Gym-style API)

Confirm `reset()` / `step()` / `state()` round-trip cleanly through the in-process `SMELiquidityEnvironment`. This mirrors `tutorial/01-environments.md` from OpenEnv.

In [ ]:
from server.environment import SMELiquidityEnvironment

env_smoke = SMELiquidityEnvironment(total_periods=TOTAL_PERIODS)
obs = env_smoke.reset(seed=42, difficulty=DIFFICULTY, task_name=TASK_NAME)
sme = env_smoke.state.world_state.smes[0]
print({
    "task_name": obs.task_name,
    "buyer_price": obs.buyer_price,
    "buyer_days": obs.buyer_days,
    "liquidity_threshold": obs.liquidity_threshold,
    "open_deals": obs.open_deal_ids,
    "initial_cash": sme.cash_balance,
    "required_minimum_cash": sme.required_minimum_cash,
    "current_period": f"{obs.current_period}/{obs.total_periods}",
})

## 4. Reproducible heuristic baseline on all three tasks

These deterministic baseline numbers are the “before” side of the judging criterion *Showing Improvement in Rewards (20%)*.

In [ ]:
from rl.demo import run_heuristic_episode

BASELINE_TASKS = [
    ("payment-terms-easy",        "easy"),
    ("payment-terms-medium",      "medium"),
    ("liquidity-correlation-hard", "hard"),
]
BASELINE_SEEDS = [11, 22, 33]

baseline = {}
for task, diff in BASELINE_TASKS:
    runs = [run_heuristic_episode(seed=s, total_periods=TOTAL_PERIODS, task_name=task, difficulty=diff) for s in BASELINE_SEEDS]
    avg = sum(r["summary"]["verifiable_reward"] for r in runs) / len(runs)
    baseline[task] = round(avg, 4)
    print(f"  baseline {task:30s} avg_verifiable_reward={avg:.4f}")

## 5. TRL/vLLM Compatibility Patches + Canonical `rollout_once`

Following the [kube-sre-gym](https://github.com/sid-rp/kube-sre-gym/blob/main/train.py) pattern exactly:

1. **Patch TRL/vLLM logprob format** (`patch_trl_vllm_compat`) — TRL 0.29 expects list-of-lists logprobs but vLLM 0.11+ returns plain floats.
2. **`apply_chat_template`** with `enable_thinking=False` fallback for Qwen3 models.
3. **`rollout_once`** — resets env, multi-turn loop with `generate_rollout_completions`, conversation history, token accumulation across turns, verifiable rewards.
4. Return parallel lists for the five reward functions to read from `**kwargs`.

In [ ]:
import csv
import logging
from datetime import datetime
from typing import Any

from server.environment import SMELiquidityEnvironment
from rl.bridge import format_observation, parse_action
from sme_negotiator_env.graders import compute_verifiable_reward_breakdown
from trl import GRPOTrainer

logging.basicConfig(level=logging.INFO, format="%(asctime)s %(levelname)s %(message)s")
logger = logging.getLogger(__name__)

# ---- TRL 0.29 / vLLM 0.11.x compatibility (from kube-sre-gym) ----
# TRL expects logprobs as list-of-lists (top-k); vLLM returns plain floats.
# See: https://github.com/huggingface/trl/issues/4159

_orig_vllm_gen = None

def _patch_vllm_generate(trainer):
    """Wrap vLLM generate to ensure logprobs are in top-k list format."""
    global _orig_vllm_gen
    if _orig_vllm_gen is not None or not hasattr(trainer, 'vllm_generation'):
        return
    _orig_vllm_gen = trainer.vllm_generation.generate
    def _wrapped_generate(**kwargs):
        result = _orig_vllm_gen(**kwargs)
        prompt_ids, completion_ids, logprobs, *rest = result
        if logprobs and logprobs[0] and isinstance(logprobs[0][0], float):
            logprobs = [[[lp] for lp in seq] for seq in logprobs]
        return (prompt_ids, completion_ids, logprobs, *rest)
    trainer.vllm_generation.generate = _wrapped_generate

def patch_trl_vllm_compat():
    """Apply TRL/vLLM compatibility patches. Call before trainer.train()."""
    _orig_train = GRPOTrainer.train
    def _patched_train(self, *args, **kwargs):
        _patch_vllm_generate(self)
        return _orig_train(self, *args, **kwargs)
    GRPOTrainer.train = _patched_train

patch_trl_vllm_compat()

# ---- Chat template with Qwen3 fallback (from kube-sre-gym) ----
def apply_chat_template(tokenizer, messages):
    """Apply chat template with fallback if enable_thinking is not supported."""
    try:
        return tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False, enable_thinking=False,
        )
    except TypeError:
        return tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=False,
        )

# ---- System prompt ----
SYSTEM_PROMPT = (
    "You are an SME treasury agent operating a long-horizon liquidity workflow.\n"
    "On every turn reply with ONE JSON action object. No prose outside the JSON.\n\n"
    "Schema:\n"
    '{"action_type": "propose"|"accept"|"reject"|"tool"|"advance_period",\n'
    ' "deal_id": "<from open_deal_ids>",\n'
    ' "price": <float, >= cost_threshold>,\n'
    ' "payment_days": <int, target <= liquidity_threshold>,\n'
    ' "use_treds": true|false,\n'
    ' "propose_late_payment_penalty_clause": true|false,\n'
    ' "reason": "<short rationale citing numbers>"}\n\n'
    "STRATEGY: Reduce payment_days aggressively toward liquidity_threshold while keeping\n"
    "price HIGH (>= 90% of buyer_price). NPV = price * volume / (1+r)^(days/365).\n"
    "Lower days helps NPV. Lower price hurts NPV. Balance both.\n"
    "Use tools (QUERY_TREDS, CHECK_COMPLIANCE) when tenor risk is unclear.\n"
    "Advance period ONLY when no open deals remain."
)


def _build_user_prompt(observation: Any, history: list[dict] = None) -> str:
    """Build the user prompt with optional conversation history."""
    parts = []
    if history:
        parts.append("PREVIOUS ACTIONS AND RESULTS:")
        for entry in history[-5:]:
            action_text = entry.get("action", "")
            result_text = entry.get("result", "")
            if len(result_text) > 300:
                result_text = result_text[:300] + "... (truncated)"
            parts.append(f"  Action: {action_text}")
            parts.append(f"  Result: {result_text}")
        parts.append("---")
    parts.append(f"Current observation:\n{format_observation(observation)}")
    parts.append("\nReply with ONE JSON action object now.")
    return "\n".join(parts)


def _format_score(text: str) -> float:
    """Bounded text-quality reward in [0, 1] — guarantees per-group GRPO variance."""
    raw = (text or "").strip()
    if not raw:
        return 0.0
    score = 0.0
    if raw.startswith("{") and raw.rstrip().endswith("}"):
        score += 0.30
    if '"action_type"' in raw:
        score += 0.15
    for verb in ("propose", "accept", "reject", "advance_period", "tool"):
        if verb in raw.lower():
            score += 0.05
            break
    for key in ("payment_days", "use_treds", "price"):
        if key in raw.lower():
            score += 0.07
    if '"reason"' in raw:
        score += 0.10
    if 50 <= len(raw) <= 600:
        score += 0.10
    return float(min(1.0, max(0.0, score)))


def rollout_once(trainer, tokenizer, dataset_prompt: str) -> dict:
    """Run one full liquidity episode driven by the model. Returns GRPO-ready signals.

    Follows the kube-sre-gym pattern: multi-turn loop with generate_rollout_completions,
    conversation history tracking, token accumulation across turns, episode-level rewards.
    """
    from trl.experimental.openenv import generate_rollout_completions
    from sme_negotiator_env.models import NegotiationAction

    env = SMELiquidityEnvironment(total_periods=TOTAL_PERIODS)
    observation = env.reset(
        seed=hash(dataset_prompt) % (2**31),
        difficulty=DIFFICULTY,
        task_name=TASK_NAME,
    )

    prompt_ids:     list[int]   = []
    completion_ids: list[int]   = []
    logprobs:       list[float] = []
    conversation_history: list[dict] = []
    step_rewards: list[float] = []
    last_text = ""

    for _turn in range(MAX_TURNS):
        if observation.done:
            break
        if len(completion_ids) >= MAX_TOTAL_TOKENS:
            break

        # Auto-advance when no open deals remain
        if not observation.open_deal_ids and observation.current_period < observation.total_periods:
            observation = env.step(NegotiationAction(action_type="advance_period"))
            continue

        # Build prompt with history (same pattern as kube-sre-gym)
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": _build_user_prompt(observation, conversation_history)},
        ]
        prompt_text = apply_chat_template(tokenizer, messages)

        # Generate with vLLM via TRL (canonical OpenEnv pattern)
        rollout = generate_rollout_completions(trainer, [prompt_text])[0]
        prompt_ids.extend(rollout["prompt_ids"])
        completion_ids.extend(rollout["completion_ids"])
        logprobs.extend(rollout["logprobs"])
        completion_text = rollout.get("text") or tokenizer.decode(
            rollout["completion_ids"], skip_special_tokens=True
        )
        last_text = completion_text

        # Parse and execute action
        try:
            action = parse_action(completion_text, observation)
            prev_obs_text = format_observation(observation)[:200]
            observation = env.step(action)
            reward = float(observation.reward or 0.0)
            step_rewards.append(reward)
            conversation_history.append({
                "action": completion_text[:300],
                "result": format_observation(observation)[:300],
                "reward": reward,
            })
        except Exception as e:
            step_rewards.append(-0.5)
            conversation_history.append({
                "action": completion_text[:200],
                "result": f"ERROR: {e}",
                "reward": -0.5,
            })
            observation = env.step(NegotiationAction(action_type="advance_period"))

    # Derive verifiable rewards from the env's WorldState across all resolved deals
    state = env.state
    world_state = state.world_state
    weighted = {"solvency": 0.0, "liquidity": 0.0, "npv": 0.0, "compliance": 0.0, "total": 0.0}
    weight_sum = 0.0
    for deal in world_state.deals:
        if deal.status == "open":
            continue
        traj = state.deal_trajectories.get(deal.deal_id) or []
        if not traj:
            continue
        bd = compute_verifiable_reward_breakdown(world_state, list(traj))
        w = float(deal.invoice_amount) if float(deal.invoice_amount) > 0 else 1.0
        weighted["solvency"]   += w * float(bd.solvency)
        weighted["liquidity"]  += w * float(bd.liquidity)
        weighted["npv"]        += w * float(bd.npv)
        weighted["compliance"] += w * float(bd.compliance)
        weighted["total"]      += w * float(bd.total)
        weight_sum += w
    if weight_sum > 0:
        for k in weighted:
            weighted[k] = round(weighted[k] / weight_sum, 6)
    defaulted = any(s.defaulted for s in world_state.smes)
    if defaulted:
        for k in weighted:
            weighted[k] = 0.0

    # Combine verifiable + format signal (format provides per-group variance)
    format_reward = _format_score(last_text)
    total_with_format = float(weighted["total"]) + 0.1 * format_reward

    return {
        "prompt_ids":        prompt_ids,
        "completion_ids":    completion_ids,
        "logprobs":          logprobs,
        "reward_total":      total_with_format,
        "reward_solvency":   float(weighted["solvency"]),
        "reward_npv":        float(weighted["npv"]),
        "reward_compliance": float(weighted["compliance"]),
        "reward_format":     format_reward,
    }

print(f"System prompt (first 200 chars):\n{SYSTEM_PROMPT[:200]}...")
print(f"\nImported: rollout_once, apply_chat_template, patch_trl_vllm_compat")

## 6. Multiple verifiable reward functions (canonical TRL pattern)

Following the *Reward Engineering* guidance (https://arxiv.org/abs/2408.10215, https://arxiv.org/abs/2410.19100):

* `reward_total`     — the canonical 0.35/0.20/0.35/0.10 RLVR composite (dominant signal).
* `reward_solvency`  — anti-default safety check (penalises destructive actions).
* `reward_npv`       — NPV uplift vs the SME's status-quo baseline.
* `reward_format`    — bounded text-quality term that guarantees per-group variance so GRPO advantage ≠ 0 from step 1.

Reward weights match the kube-sre-gym style: outcome reward dominant, format reward bounded.

In [ ]:
def reward_total(completions, **kwargs):
    rewards = kwargs.get("reward_total")
    return [float(r) for r in rewards] if rewards is not None else [0.0 for _ in completions]

def reward_solvency(completions, **kwargs):
    rewards = kwargs.get("reward_solvency")
    return [float(r) for r in rewards] if rewards is not None else [0.0 for _ in completions]

def reward_npv(completions, **kwargs):
    rewards = kwargs.get("reward_npv")
    return [float(r) for r in rewards] if rewards is not None else [0.0 for _ in completions]

def reward_compliance(completions, **kwargs):
    rewards = kwargs.get("reward_compliance")
    return [float(r) for r in rewards] if rewards is not None else [0.0 for _ in completions]

def reward_format(completions, **kwargs):
    rewards = kwargs.get("reward_format")
    return [float(r) for r in rewards] if rewards is not None else [0.0 for _ in completions]

REWARD_FUNCS   = [reward_total, reward_solvency, reward_npv, reward_compliance, reward_format]
REWARD_WEIGHTS = [1.0,          0.3,             0.3,        0.2,               0.1]

## 7. Build dataset, tokenizer, and `GRPOConfig`

Same shape as the canonical Wordle script (`tutorial/examples/wordle.py`).

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer
from trl import GRPOConfig
import torch

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "left"

# Dataset — each row triggers one episode (content is a length driver, not used in rollout_once)
DATASET_PROMPT = "Negotiate the SME's payment terms across the macro horizon while staying solvent."
dataset = Dataset.from_dict({"prompt": [DATASET_PROMPT] * DATASET_SIZE})

torch_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else (torch.float16 if torch.cuda.is_available() else None)

# Reward CSV logger (same pattern as kube-sre-gym)
reward_log_path = OUTPUT_DIR / "reward_log.csv"
episode_counter = [0]
all_episode_rewards = []

with open(reward_log_path, "w", newline="") as f:
    csv.writer(f).writerow(["episode", "total_reward", "solvency", "npv", "compliance", "format", "timestamp"])

def log_episode(total_r, solv_r, npv_r, comp_r, fmt_r):
    episode_counter[0] += 1
    all_episode_rewards.append(total_r)
    with open(reward_log_path, "a", newline="") as f:
        csv.writer(f).writerow([episode_counter[0], total_r, solv_r, npv_r, comp_r, fmt_r, datetime.now().isoformat()])
    last10 = all_episode_rewards[-10:]
    logger.info(
        f"Episode {episode_counter[0]}: reward={total_r:.3f} | "
        f"mean={sum(all_episode_rewards)/len(all_episode_rewards):.3f}, "
        f"mean(10)={sum(last10)/len(last10):.3f}"
    )

grpo_config = GRPOConfig(
    use_vllm=True,
    vllm_mode="colocate",
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=1,
    max_steps=TRAIN_STEPS,
    learning_rate=5e-6,
    weight_decay=0.0,
    warmup_steps=2,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    generation_batch_size=NUM_GENERATIONS,
    num_generations=NUM_GENERATIONS,
    max_completion_length=MAX_NEW_TOKENS,
    temperature=0.9,
    logging_steps=1,
    save_strategy="steps",
    save_steps=max(1, TRAIN_STEPS),
    save_total_limit=2,
    report_to="none",
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=(torch_dtype == torch.bfloat16),
    fp16=(torch_dtype == torch.float16),
    reward_weights=REWARD_WEIGHTS,
)
print("GRPOConfig ready. dtype=", torch_dtype)

## 8. Wire `rollout_func` and Train

The `rollout_func` produces parallel lists for the five reward functions exactly like `kube-sre-gym/train.py` and `tutorial/examples/wordle.py`. Each row triggers one episode via `rollout_once`.

Key architecture (follows kube-sre-gym):
- **`GRPOTrainer(model=MODEL_NAME, rollout_func=rollout_func, peft_config=peft_config)`** — TRL handles vLLM colocate + model loading
- **PEFT/LoRA** with rank=16 — fits a free-tier T4
- **Per-episode CSV logging** for reward curves
- **`patch_trl_vllm_compat()`** applied to fix logprob format

In [ ]:
from transformers import TrainerCallback
from trl import GRPOTrainer
from peft import LoraConfig

# ---- LoRA Config (same as kube-sre-gym) ----
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)


# ---- rollout_func (canonical TRL pattern from kube-sre-gym) ----
def rollout_func(prompts, trainer):
    """Per-prompt rollout returning parallel reward lists (nested lists for TRL)."""
    out = {
        "prompt_ids":        [],
        "completion_ids":    [],
        "logprobs":          [],
        "reward_total":      [],
        "reward_solvency":   [],
        "reward_npv":        [],
        "reward_compliance": [],
        "reward_format":     [],
    }
    for prompt_text in prompts:
        ep = rollout_once(trainer, tokenizer, prompt_text if isinstance(prompt_text, str) else str(prompt_text))
        out["prompt_ids"].append(ep["prompt_ids"])
        out["completion_ids"].append(ep["completion_ids"])
        out["logprobs"].append(ep["logprobs"])
        out["reward_total"].append(ep["reward_total"])
        out["reward_solvency"].append(ep["reward_solvency"])
        out["reward_npv"].append(ep["reward_npv"])
        out["reward_compliance"].append(ep["reward_compliance"])
        out["reward_format"].append(ep["reward_format"])
        log_episode(ep["reward_total"], ep["reward_solvency"], ep["reward_npv"], ep["reward_compliance"], ep["reward_format"])
    return out


# ---- Reward curve callback ----
class RewardCurveCallback(TrainerCallback):
    """Capture per-step (avg) reward for the post-training curve."""
    def __init__(self):
        self.steps = []
        self.avg_total = []
        self.avg_solvency = []
        self.avg_npv = []
        self.avg_compliance = []
        self.avg_format = []
        self.loss = []
    def on_log(self, args, state, control, logs=None, **kwargs):
        if not logs:
            return control
        if "loss" in logs:
            self.loss.append(float(logs["loss"]))
        for key, dest in (
            ("rewards/reward_total/mean",      self.avg_total),
            ("rewards/reward_solvency/mean",   self.avg_solvency),
            ("rewards/reward_npv/mean",        self.avg_npv),
            ("rewards/reward_compliance/mean", self.avg_compliance),
            ("rewards/reward_format/mean",     self.avg_format),
        ):
            if key in logs:
                dest.append(float(logs[key]))
        if any(k.startswith("rewards/") for k in logs):
            self.steps.append(int(state.global_step))
        return control

reward_callback = RewardCurveCallback()


# ---- Create trainer (same shape as kube-sre-gym) ----
trainer = GRPOTrainer(
    model=MODEL_NAME,
    processing_class=tokenizer,
    reward_funcs=REWARD_FUNCS,
    train_dataset=dataset,
    args=grpo_config,
    rollout_func=rollout_func,
    peft_config=peft_config,
    callbacks=[reward_callback],
)

print("GRPOTrainer initialized")
print(f"  Model       : {MODEL_NAME}")
print(f"  Episodes    : {DATASET_SIZE}")
print(f"  Generations : {NUM_GENERATIONS}")
print(f"  Max turns   : {MAX_TURNS}")
print(f"  Steps       : {TRAIN_STEPS}")
print()
print("Starting GRPO training...")
trainer.train()

checkpoint_path = OUTPUT_DIR / "final-grpo-model"
trainer.save_model(str(checkpoint_path))
tokenizer.save_pretrained(str(checkpoint_path))
print(f"\nCheckpoint saved to {checkpoint_path}")

## 9. Reward curves + baseline overlay

Two plots, both labelled with units, both saved as `.png` so judges can pull them straight from the repo.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# ---- Plot 1: Per-episode rewards from CSV log (same as kube-sre-gym) ----
episodes_csv, totals_csv = [], []
try:
    with open(reward_log_path) as f:
        reader = csv.reader(f)
        next(reader)
        for row in reader:
            episodes_csv.append(int(row[0]))
            totals_csv.append(float(row[1]))
except Exception:
    pass

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Episode reward curve with rolling average
ax1 = axes[0, 0]
if episodes_csv:
    window = min(10, len(episodes_csv))
    rolling = [sum(totals_csv[max(0,i-window):i+1]) / min(i+1, window) for i in range(len(totals_csv))]
    ax1.plot(episodes_csv, totals_csv, alpha=0.25, color="blue", marker="o", markersize=3, label="Per episode")
    ax1.plot(episodes_csv, rolling, color="blue", linewidth=2.5, label=f"Rolling avg ({window})")
    z = np.polyfit(episodes_csv, totals_csv, 1)
    trend = np.poly1d(z)
    ax1.plot(episodes_csv, trend(episodes_csv), color="red", linewidth=1.5, linestyle="--",
             label=f"Trend ({'↑' if z[0] > 0 else '↓'} {abs(z[0]):.4f}/ep)")
ax1.axhline(y=baseline.get(TASK_NAME, 0.0), color="green", linestyle="--", linewidth=1.5, label=f"Baseline")
ax1.set_ylabel("Total Reward")
ax1.set_xlabel("Episode")
ax1.set_title("SME Negotiator — Per-Episode Reward")
ax1.legend(fontsize=7)
ax1.grid(alpha=0.3)

# GRPO step reward components
ax2 = axes[0, 1]
if reward_callback.steps:
    ax2.plot(reward_callback.steps, reward_callback.avg_total,      label="total (RLVR)", linewidth=2)
    ax2.plot(reward_callback.steps, reward_callback.avg_solvency,   label="solvency")
    ax2.plot(reward_callback.steps, reward_callback.avg_npv,        label="npv")
    ax2.plot(reward_callback.steps, reward_callback.avg_compliance, label="compliance")
    ax2.plot(reward_callback.steps, reward_callback.avg_format,     label="format", linestyle=":")
ax2.set_ylabel("Avg Reward (per GRPO step)")
ax2.set_xlabel("GRPO Step")
ax2.set_title("Reward Components Over Training")
ax2.legend(fontsize=7)
ax2.grid(alpha=0.3)

# Loss curve
ax3 = axes[1, 0]
if reward_callback.loss:
    ax3.plot(range(1, len(reward_callback.loss) + 1), reward_callback.loss, label="GRPO loss", color="black")
ax3.set_xlabel("GRPO Step")
ax3.set_ylabel("Loss")
ax3.set_title("Training Loss")
ax3.legend(fontsize=7)
ax3.grid(alpha=0.3)

# Summary stats
ax4 = axes[1, 1]
ax4.axis("off")
if totals_csv:
    stats_text = (
        f"Total Episodes: {len(totals_csv)}\n"
        f"Mean Reward: {sum(totals_csv)/len(totals_csv):.4f}\n"
        f"Best Reward: {max(totals_csv):.4f}\n"
        f"Final Avg (10): {sum(totals_csv[-10:])/min(10,len(totals_csv)):.4f}\n"
        f"Baseline: {baseline.get(TASK_NAME, 0.0):.4f}\n"
        f"Training Steps: {TRAIN_STEPS}\n"
        f"GRPO Loss Steps: {len(reward_callback.loss)}\n"
    )
    non_zero_loss = sum(1 for l in reward_callback.loss if l > 1e-8)
    stats_text += f"Non-zero Loss Steps: {non_zero_loss}/{len(reward_callback.loss)}"
    ax4.text(0.1, 0.5, stats_text, transform=ax4.transAxes, fontsize=11,
             verticalalignment="center", fontfamily="monospace",
             bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
    ax4.set_title("Training Summary")

fig.suptitle("SME Negotiator — GRPO Training (Theme #1: Multi-Agent Interactions)", fontsize=13, fontweight="bold")
fig.tight_layout()

curve_path = OUTPUT_DIR / "reward_curve.png"
fig.savefig(curve_path, dpi=140, bbox_inches="tight")
plt.show()
print(f"Reward curve saved to {curve_path}")

## 10. Before / after evaluation — trained policy vs heuristic baseline

The model is loaded back via `peft.PeftModel` so the LoRA adapter is actually applied (this matters — see OpenEnv tip #16, never naively merge a LoRA adapter).

In [ ]:
from transformers import AutoModelForCausalLM
import torch

try:
    from peft import PeftModel
    base = AutoModelForCausalLM.from_pretrained(MODEL_NAME, torch_dtype=torch_dtype, low_cpu_mem_usage=True)
    trained_model = PeftModel.from_pretrained(base, str(checkpoint_path))
    trained_model.eval()
except Exception as exc:
    print("[warn] PeftModel load failed; falling back to plain HF load:", exc)
    trained_model = AutoModelForCausalLM.from_pretrained(str(checkpoint_path), torch_dtype=torch_dtype)
    trained_model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
trained_model.to(device)


def evaluate_trained_policy(seed: int = 7) -> dict:
    env = SMELiquidityEnvironment(total_periods=TOTAL_PERIODS)
    observation = env.reset(seed=seed, difficulty=DIFFICULTY, task_name=TASK_NAME)
    for _ in range(MAX_TURNS):
        if observation.done:
            break
        if not observation.open_deal_ids and observation.current_period < observation.total_periods:
            from sme_negotiator_env.models import NegotiationAction
            observation = env.step(NegotiationAction(action_type="advance_period"))
            continue
        chat = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": _build_user_prompt(observation)},
        ]
        prompt_text = tokenizer.apply_chat_template(chat, add_generation_prompt=True, tokenize=False)
        inputs = tokenizer(prompt_text, return_tensors="pt", truncation=True, max_length=1400).to(device)
        with torch.no_grad():
            generated = trained_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        completion = tokenizer.decode(generated[0, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        try:
            action = parse_action(completion, observation)
            observation = env.step(action)
        except Exception:
            from sme_negotiator_env.models import NegotiationAction
            observation = env.step(NegotiationAction(action_type="advance_period"))

    state = env.state
    world = state.world_state
    weighted_total = 0.0
    weight_sum = 0.0
    for deal in world.deals:
        if deal.status == "open":
            continue
        traj = state.deal_trajectories.get(deal.deal_id) or []
        if not traj:
            continue
        bd = compute_verifiable_reward_breakdown(world, list(traj))
        w = float(deal.invoice_amount) if float(deal.invoice_amount) > 0 else 1.0
        weighted_total += w * float(bd.total)
        weight_sum += w
    return {
        "trained_avg_verifiable_reward": round(weighted_total / max(weight_sum, 1.0), 6),
        "defaulted_sme_count":           sum(1 for s in world.smes if s.defaulted),
        "resolved_deal_count":           len(state.resolved_deal_ids),
    }

trained_summary = evaluate_trained_policy(seed=7)
print("BEFORE (heuristic baseline) :", baseline)
print("AFTER  (trained policy)     :", trained_summary)

## 11. (Optional) Publish checkpoint + model card to the Hugging Face Hub

Set `HF_REPO_ID` to your namespace, then run.

In [ ]:
from huggingface_hub import HfApi
from pathlib import Path

HF_REPO_ID = "YOUR_USERNAME/openenv-sme-negotiator-grpo"
CARD_PATH  = Path("huggingface/model_card.md")

if not HF_REPO_ID.startswith("YOUR_USERNAME/") and CARD_PATH.exists():
    api = HfApi()
    api.create_repo(repo_id=HF_REPO_ID, repo_type="model", exist_ok=True)
    api.upload_folder(
        repo_id=HF_REPO_ID,
        repo_type="model",
        folder_path=str(checkpoint_path),
        ignore_patterns=["*.tmp", "*.log"],
    )
    api.upload_file(
        repo_id=HF_REPO_ID,
        repo_type="model",
        path_or_fileobj=str(CARD_PATH),
        path_in_repo="README.md",
    )
    print(f"published https://huggingface.co/{HF_REPO_ID}")
else:
    print("Skipping Hub publish (HF_REPO_ID not set or model_card.md missing).")